# Verify noise augmentation

Confirm `NoiseMixer` produces sane, reproducible noisy audio across every
SNR bucket and noise category before trusting it in the training loop or
the reliability head's supervision signal.

**What "looks right" means:**
- `mixer.available_categories()` excludes the held-out category
  (`"noise"` by default) unless `include_held_out=True` is passed.
- For each non-"clean" SNR bucket, the ACHIEVED SNR of the mixed output
  (measured the same way `_mix_at_snr` is unit-tested: signal power vs.
  added-noise power, in dB) is close to the target bucket -- some slack
  is expected on short/quiet real clips (unlike the unit test's synthetic
  Gaussian signals, which match to ~0.1dB), but it should be in the right
  ballpark, not off by many dB.
- `snr_bucket="clean"` returns the input completely unchanged.
- Mixing the same clean audio with the same seed twice produces BIT-FOR-BIT
  identical output -- this is the reproducibility guarantee `DEFAULT_SEED
  = 42` exists for.
- The mixed waveforms look qualitatively right when plotted (visibly
  noisier at 0dB/-5dB than at 20dB).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf

from fusion_avsr.data.noise_augmentation import NOISE_CATEGORIES, SNR_BUCKETS, NoiseMixer

## Config

In [ ]:
# TODO: fill in the real MUSAN root, and a real clean .wav path (e.g. from notebook 01/02)
MUSAN_ROOT = Path("/scratch/project_2020712/datasets/musan_noise/datasets/nhattruongdev/musan-noise/versions/1/musan")
SAMPLE_CLEAN_WAV_PATH = Path("/scratch/project_2020712/datasets/extracted_audio/REPLACE_ME.wav")

In [ ]:
clean_audio, sample_rate = sf.read(str(SAMPLE_CLEAN_WAV_PATH), dtype="float32")
assert sample_rate == 16000

# seed defaults to DEFAULT_SEED (42) -- reproducible mixing out of the box
mixer = NoiseMixer(musan_root=MUSAN_ROOT, sample_rate=sample_rate)

## Held-out category behavior

In [ ]:
print("available for training (held-out excluded):", mixer.available_categories())
print("available for eval (held-out included):     ", mixer.available_categories(include_held_out=True))

assert "noise" not in mixer.available_categories()
assert set(mixer.available_categories(include_held_out=True)) == set(NOISE_CATEGORIES)

## Achieved SNR per bucket/category

In [ ]:
def achieved_snr_db(clean, mixed):
    added_noise = mixed - clean
    signal_power = np.mean(clean ** 2)
    added_noise_power = np.mean(added_noise ** 2)
    return 10 * np.log10(signal_power / added_noise_power)


results = []
for category in NOISE_CATEGORIES:  # include the held-out category too, for this eval-style check
    for bucket in SNR_BUCKETS:
        if bucket == "clean":
            continue
        mixed = mixer.mix(clean_audio, snr_bucket=bucket, category=category)
        results.append({
            "category": category,
            "target_snr_db": bucket,
            "achieved_snr_db": round(achieved_snr_db(clean_audio, mixed), 2),
        })

import pandas as pd
pd.DataFrame(results)

## Clean bucket is a no-op

In [ ]:
mixed_clean = mixer.mix(clean_audio, snr_bucket="clean")
assert np.array_equal(mixed_clean, clean_audio)
print("clean bucket returns input unchanged: OK")

## Reproducibility: same seed + same input -> identical output

In [ ]:
mixer_a = NoiseMixer(musan_root=MUSAN_ROOT, sample_rate=sample_rate, seed=42)
mixer_b = NoiseMixer(musan_root=MUSAN_ROOT, sample_rate=sample_rate, seed=42)

out_a = mixer_a.mix(clean_audio, snr_bucket=5, category="music")
out_b = mixer_b.mix(clean_audio, snr_bucket=5, category="music")

assert np.array_equal(out_a, out_b), "same seed should give bit-for-bit identical output"
print("reproducibility check: OK")

## Visual sanity check: waveforms across SNR buckets

In [ ]:
buckets_to_plot = [20, 5, -5]
fig, axes = plt.subplots(len(buckets_to_plot) + 1, 1, figsize=(10, 8), sharex=True)

axes[0].plot(clean_audio)
axes[0].set_title("clean")

for ax, bucket in zip(axes[1:], buckets_to_plot):
    mixed = mixer.mix(clean_audio, snr_bucket=bucket, category="music")
    ax.plot(mixed)
    ax.set_title(f"music @ {bucket}dB")

plt.tight_layout()
plt.show()

## TODO checklist

- [ ] `available_categories()` excludes `"noise"` by default.
- [ ] Achieved SNR table above is in the right ballpark per bucket (not
      exact, but not wildly off either).
- [ ] `clean` bucket assertion passed.
- [ ] Reproducibility assertion passed.
- [ ] Waveform plots visibly get noisier as target SNR drops.